In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression


In [1]:
df = pd.read_csv("../../DataSets/train.csv")

NameError: name 'pd' is not defined

In [ ]:
df.head()

In [ ]:
df.drop(columns=["PassengerId", "Name", "Ticket","Cabin"], inplace=True)

In [ ]:
df.head()

In [ ]:
X = df.drop(columns=["Survived"])
y = df["Survived"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 2)

In [ ]:
X_train.head()

In [ ]:
numerical_feature = ["Age","Fare"]

numerical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy="median")),
    ('scaler', StandardScaler())
])

categorical_feature = ["Embarked","Sex"]
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy="most_frequent")),
    ('onehot', OneHotEncoder(handle_unknown="ignore"))
])

In [ ]:
preprocessor = ColumnTransformer([
    ('num', numerical_transformer, numerical_feature),
    ('cat', categorical_transformer, categorical_feature)
])

In [ ]:
clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])

In [ ]:
clf

In [ ]:
param_grid = {
    'preprocessor__num__imputer__strategy': ['mean','median'],
    "preprocessor__cat__imputer__strategy" : ["most_frequent","constant"],
    "classifier__C" : [0.1,1.0,10,100]
}

grid_search = GridSearchCV(clf, param_grid=param_grid, cv=10)

In [ ]:
grid_search.fit(X_train, y_train)

print("Best params:")
print(grid_search.best_params_)

In [ ]:
print(f"Internal CVF Score: {grid_search.best_score_:.3f}")

In [ ]:
res = pd.DataFrame(grid_search.cv_results_)
res.sort_values("mean_test_score", ascending=False)